In [4]:
import pandas as pd
import os

# ==========================================
# 1. パス設定 (絶対パスで指定)
# ==========================================
RUN_ID = "20251130_153500"
ROOT_BASE = "/Volumes/csbdeep11/_yasu-i/20250303_rebuiled/cal_cluster_No/20251026_under_25clusters_program_and_result/for_checking_20251127"

# 各フォルダのパスを構築
DIR_THESIS = os.path.join(ROOT_BASE, "Thesis_Figures", f"run_{RUN_ID}")
DIR_LABELS = os.path.join(ROOT_BASE, "sub", "04_summary_STEP3.2_signlessCorr", f"run_{RUN_ID}", "samples")
DIR_DETAIL = os.path.join(ROOT_BASE, "Thesis_Figures", f"run_{RUN_ID}", "paper_4.2.4.4_OHFP", "reverse", "analysis_csv")

# 出力先 (Thesis_Figures フォルダに保存)
OUTPUT_DIR = DIR_THESIS

print(f"Target Run ID: {RUN_ID}")
print(f"Looking for files in:")
print(f"  1. {DIR_THESIS}")
print(f"  2. {DIR_LABELS}")
print(f"  3. {DIR_DETAIL}")

# ==========================================
# 2. ファイルパスの定義
# ==========================================
# ※ファイル名が確実なものを指定

# 1. 統合データ
file_thesis_A = os.path.join(DIR_THESIS, "ThesisData_A_OH_plus_others_MDS_Targets.csv")
file_thesis_B = os.path.join(DIR_THESIS, "ThesisData_B_FP_plus_others_MDS_Targets.csv")

# 2. ラベルデータ
file_labels_A = os.path.join(DIR_LABELS, f"cluster_labels_matrix_samples_A_OH_plus_others_{RUN_ID}.csv")
file_labels_B = os.path.join(DIR_LABELS, f"cluster_labels_matrix_samples_B_FP_plus_others_{RUN_ID}.csv")

# 3. 詳細データ
file_detail_OH_k50 = os.path.join(DIR_DETAIL, "Detail_OH_Clusters_linear_top3_k50.csv")
file_detail_OH_k10 = os.path.join(DIR_DETAIL, "Detail_OH_Clusters_linear_top3_k10.csv")
file_detail_FP_k50 = os.path.join(DIR_DETAIL, "Detail_FP_Clusters_linear_top3_k50.csv")
file_detail_FP_k10 = os.path.join(DIR_DETAIL, "Detail_FP_Clusters_linear_top3_k10.csv")

# 存在確認 (デバッグ用)
for f in [file_thesis_A, file_labels_A, file_detail_OH_k50]:
    if os.path.exists(f):
        print(f"[OK] Found: {os.path.basename(f)}")
    else:
        print(f"[ERROR] Not found: {f}")

# ==========================================
# 3. データ読み込み & 結合
# ==========================================
def load_and_merge(f_thesis, f_labels):
    if not os.path.exists(f_thesis) or not os.path.exists(f_labels):
        return None
    try:
        df_t = pd.read_csv(f_thesis)
        if 'Unnamed: 0' in df_t.columns:
            df_t.rename(columns={'Unnamed: 0': 'ID'}, inplace=True)
        
        df_l = pd.read_csv(f_labels)
        # IDで結合
        df = pd.merge(df_t, df_l, on='ID', how='inner')
        return df
    except Exception as e:
        print(f"[ERROR] Merge failed: {e}")
        return None

def load_detail(path):
    return pd.read_csv(path) if os.path.exists(path) else None

print("\nLoading Data...")
df_A = load_and_merge(file_thesis_A, file_labels_A)
df_B = load_and_merge(file_thesis_B, file_labels_B)

det_OH_k50 = load_detail(file_detail_OH_k50)
det_OH_k10 = load_detail(file_detail_OH_k10)
det_FP_k50 = load_detail(file_detail_FP_k50)
det_FP_k10 = load_detail(file_detail_FP_k10)

# ==========================================
# 4. 集計ロジック
# ==========================================
def get_stats(df, col_k, cluster_id, detail_df, type_name):
    if df is None: return None
    if col_k not in df.columns: return None
    
    subset = df[df[col_k] == cluster_id]
    if subset.empty: return None
        
    pce_data = subset['PCEmax']
    
    desc = "N/A"
    if detail_df is not None:
        if 'Cluster_ID' in detail_df.columns:
            row = detail_df[detail_df['Cluster_ID'] == cluster_id]
            if not row.empty:
                col_comp = 'Major_Material' if type_name == 'OH' else 'Major_Fragment'
                if col_comp in row.columns:
                    raw_name = str(row[col_comp].values[0])
                    desc = raw_name.replace("SMILESsname", "").replace("p1M_", "").replace("nM_", "").replace("inM_", "")
    
    return {
        'Count': len(subset),
        'Mean_PCE': pce_data.mean(),
        'Std_Dev': pce_data.std(),
        'Max_PCE': pce_data.max(),
        'Major_Component': desc
    }

# ==========================================
# 5. テーブル作成実行
# ==========================================

# --- A. Table 4.x (Sweet Spot Summary) ---
if df_A is not None and df_B is not None:
    print("\nGenerating Table 4.x (Sweet Spot Summary)...")
    rows = []
    
    # 1. OH k=10 Best
    col = 'linear_top3_k10'
    if col in df_A.columns:
        best_id = df_A.groupby(col)['PCEmax'].mean().idxmax()
        st = get_stats(df_A, col, best_id, det_OH_k10, 'OH')
        if st: rows.append(['OH (Material)', 10, best_id, st['Count'], st['Mean_PCE'], st['Std_Dev'], st['Max_PCE'], st['Major_Component']])
    
    # 2. OH k=50 Best
    col = 'linear_top3_k50'
    if col in df_A.columns:
        best_id = 19 # Identified previously
        st = get_stats(df_A, col, best_id, det_OH_k50, 'OH')
        if st: rows.append(['OH (Material)', 50, best_id, st['Count'], st['Mean_PCE'], st['Std_Dev'], st['Max_PCE'], st['Major_Component']])
    
    # 3. FP k=10 Best
    col = 'linear_top3_k10'
    if col in df_B.columns:
        best_id = df_B.groupby(col)['PCEmax'].mean().idxmax()
        st = get_stats(df_B, col, best_id, det_FP_k10, 'FP')
        if st: rows.append(['FP (Structure)', 10, best_id, st['Count'], st['Mean_PCE'], st['Std_Dev'], st['Max_PCE'], st['Major_Component']])
    
    # 4. FP k=50 Best
    col = 'linear_top3_k50'
    if col in df_B.columns:
        best_id = 18 # Identified previously
        st = get_stats(df_B, col, best_id, det_FP_k50, 'FP')
        if st: rows.append(['FP (Structure)', 50, best_id, st['Count'], st['Mean_PCE'], st['Std_Dev'], st['Max_PCE'], st['Major_Component']])
    
    cols = ['Dataset', 'Resolution (k)', 'Cluster ID', 'N', 'Mean PCE (%)', 'Std Dev', 'Max PCE (%)', 'Major Component']
    df_main = pd.DataFrame(rows, columns=cols)
    
    out_path = os.path.join(OUTPUT_DIR, 'Table_4_x_Sweet_Spot_Summary.csv')
    df_main.to_csv(out_path, index=False)
    print(f"  [SAVE] {out_path}")
    print(df_main)

# --- B. Appendix Tables (Full Ranking) ---
def generate_ranking(df, detail_df, type_name, out_name):
    col_k = 'linear_top3_k50'
    if df is None or col_k not in df.columns: return

    print(f"\nGenerating Full Ranking for {type_name}...")
    cids = sorted(df[col_k].unique())
    rr = []
    for cid in cids:
        st = get_stats(df, col_k, cid, detail_df, type_name)
        if st:
            rr.append([cid, st['Count'], st['Mean_PCE'], st['Std_Dev'], st['Max_PCE'], st['Major_Component']])
    
    df_rank = pd.DataFrame(rr, columns=['Cluster ID', 'N', 'Mean PCE (%)', 'Std Dev', 'Max PCE (%)', 'Major Component'])
    df_rank = df_rank.sort_values('Mean PCE (%)', ascending=False)
    
    out_path = os.path.join(OUTPUT_DIR, out_name)
    df_rank.to_csv(out_path, index=False)
    print(f"  [SAVE] {out_path}")

generate_ranking(df_A, det_OH_k50, 'OH', 'Table_Appendix_OH_All_Clusters_k50.csv')
generate_ranking(df_B, det_FP_k50, 'FP', 'Table_Appendix_FP_All_Clusters_k50.csv')

print("\n✅ All Tables Generated Successfully!")

Target Run ID: 20251130_153500
Looking for files in:
  1. /Volumes/csbdeep11/_yasu-i/20250303_rebuiled/cal_cluster_No/20251026_under_25clusters_program_and_result/for_checking_20251127/Thesis_Figures/run_20251130_153500
  2. /Volumes/csbdeep11/_yasu-i/20250303_rebuiled/cal_cluster_No/20251026_under_25clusters_program_and_result/for_checking_20251127/sub/04_summary_STEP3.2_signlessCorr/run_20251130_153500/samples
  3. /Volumes/csbdeep11/_yasu-i/20250303_rebuiled/cal_cluster_No/20251026_under_25clusters_program_and_result/for_checking_20251127/Thesis_Figures/run_20251130_153500/paper_4.2.4.4_OHFP/reverse/analysis_csv
[OK] Found: ThesisData_A_OH_plus_others_MDS_Targets.csv
[OK] Found: cluster_labels_matrix_samples_A_OH_plus_others_20251130_153500.csv
[OK] Found: Detail_OH_Clusters_linear_top3_k50.csv

Loading Data...

Generating Table 4.x (Sweet Spot Summary)...
  [SAVE] /Volumes/csbdeep11/_yasu-i/20250303_rebuiled/cal_cluster_No/20251026_under_25clusters_program_and_result/for_checking_2

In [10]:
import pandas as pd
import os
import glob

# Define expected patterns and find actual files
def find_file(pattern):
    files = glob.glob(pattern)
    if files:
        return files[0]
    return None

# 1. Identify Files
file_thesis_A = find_file('*ThesisData_A_OH_plus_others_MDS_Targets*.csv')
file_thesis_B = find_file('*ThesisData_B_FP_plus_others_MDS_Targets*.csv')

file_labels_A = find_file('*cluster_labels_matrix_samples_A_OH_plus_others*.csv')
file_labels_B = find_file('*cluster_labels_matrix_samples_B_FP_plus_others*.csv')

# Detail files
file_detail_OH_k50 = 'Detail_OH_Clusters_linear_top3_k50.csv'
file_detail_OH_k10 = 'Detail_OH_Clusters_linear_top3_k10.csv'
file_detail_FP_k50 = 'Detail_FP_Clusters_linear_top3_k50.csv'
file_detail_FP_k10 = 'Detail_FP_Clusters_linear_top3_k10.csv'

print(f"Found Thesis A: {file_thesis_A}")
print(f"Found Thesis B: {file_thesis_B}")
print(f"Found Labels A: {file_labels_A}")
print(f"Found Labels B: {file_labels_B}")

# Output files
OUT_TABLE_MAIN  = 'Table_4_x_Sweet_Spot_Summary.csv'
OUT_TABLE_OH_ALL = 'Table_Appendix_OH_All_Clusters_k50.csv'
OUT_TABLE_FP_ALL = 'Table_Appendix_FP_All_Clusters_k50.csv'

# Helper to load and merge
def load_and_merge(f_thesis, f_labels):
    if not f_thesis or not f_labels or not os.path.exists(f_thesis) or not os.path.exists(f_labels):
        print(f"[WARN] Missing files for merge: {f_thesis}, {f_labels}")
        return None
    
    try:
        df_t = pd.read_csv(f_thesis)
        if 'Unnamed: 0' in df_t.columns:
            df_t.rename(columns={'Unnamed: 0': 'ID'}, inplace=True)
        
        df_l = pd.read_csv(f_labels)
        # Merge on ID
        df = pd.merge(df_t, df_l, on='ID', how='inner')
        return df
    except Exception as e:
        print(f"[ERROR] Merge failed: {e}")
        return None

def load_detail(path):
    if not os.path.exists(path):
        print(f"[WARN] Detail file not found: {path}")
        return None
    return pd.read_csv(path)

# Helper to get stats
def get_stats(df, col_k, cluster_id, detail_df, type_name):
    subset = df[df[col_k] == cluster_id]
    if subset.empty:
        return {'Count': 0, 'Mean_PCE': 0, 'Std_Dev': 0, 'Max_PCE': 0, 'Major_Component': "N/A"}
        
    pce_data = subset['PCEmax']
    
    desc = "N/A"
    if detail_df is not None:
        row = detail_df[detail_df['Cluster_ID'] == cluster_id]
        if not row.empty:
            col_comp = 'Major_Material' if type_name == 'OH' else 'Major_Fragment'
            raw_name = str(row[col_comp].values[0])
            desc = raw_name.replace("SMILESsname", "").replace("p1M_", "").replace("nM_", "").replace("inM_", "")
    
    return {
        'Count': len(subset),
        'Mean_PCE': pce_data.mean(),
        'Std_Dev': pce_data.std(),
        'Max_PCE': pce_data.max(),
        'Major_Component': desc
    }

# Main Process
df_A = load_and_merge(file_thesis_A, file_labels_A)
df_B = load_and_merge(file_thesis_B, file_labels_B)

det_OH_k50 = load_detail(file_detail_OH_k50)
det_OH_k10 = load_detail(file_detail_OH_k10)
det_FP_k50 = load_detail(file_detail_FP_k50)
det_FP_k10 = load_detail(file_detail_FP_k10)

# --- Generate Table 4.x ---
if df_A is not None and df_B is not None:
    print("\nGenerating Table 4.x...")
    rows = []
    
    # 1. OH k=10 Best
    best_id = df_A.groupby('linear_top3_k10')['PCEmax'].mean().idxmax()
    st = get_stats(df_A, 'linear_top3_k10', best_id, det_OH_k10, 'OH')
    rows.append(['OH (Material)', 10, best_id, st['Count'], st['Mean_PCE'], st['Std_Dev'], st['Max_PCE'], st['Major_Component']])
    
    # 2. OH k=50 Best (Cluster 19)
    best_id_50 = 19
    st = get_stats(df_A, 'linear_top3_k50', best_id_50, det_OH_k50, 'OH')
    rows.append(['OH (Material)', 50, best_id_50, st['Count'], st['Mean_PCE'], st['Std_Dev'], st['Max_PCE'], st['Major_Component']])
    
    # 3. FP k=10 Best
    best_id = df_B.groupby('linear_top3_k10')['PCEmax'].mean().idxmax()
    st = get_stats(df_B, 'linear_top3_k10', best_id, det_FP_k10, 'FP')
    rows.append(['FP (Structure)', 10, best_id, st['Count'], st['Mean_PCE'], st['Std_Dev'], st['Max_PCE'], st['Major_Component']])
    
    # 4. FP k=50 Best (Cluster 18)
    best_id_50_fp = 18
    st = get_stats(df_B, 'linear_top3_k50', best_id_50_fp, det_FP_k50, 'FP')
    rows.append(['FP (Structure)', 50, best_id_50_fp, st['Count'], st['Mean_PCE'], st['Std_Dev'], st['Max_PCE'], st['Major_Component']])
    
    df_table4x = pd.DataFrame(rows, columns=['Dataset', 'Resolution (k)', 'Cluster ID', 'N', 'Mean PCE (%)', 'Std Dev', 'Max PCE (%)', 'Major Component'])
    df_table4x.to_csv(OUT_TABLE_MAIN, index=False)
    print(df_table4x)
    
    # --- Generate Appendix Tables (Full Ranking) ---
    
    # OH Ranking
    if det_OH_k50 is not None:
        cids = sorted(df_A['linear_top3_k50'].unique())
        rr = []
        for cid in cids:
            st = get_stats(df_A, 'linear_top3_k50', cid, det_OH_k50, 'OH')
            rr.append([cid, st['Count'], st['Mean_PCE'], st['Std_Dev'], st['Max_PCE'], st['Major_Component']])
        df_r_oh = pd.DataFrame(rr, columns=['Cluster ID', 'N', 'Mean PCE (%)', 'Std Dev', 'Max PCE (%)', 'Major Component'])
        # Corrected sort column name (using correct space)
        df_r_oh = df_r_oh.sort_values('Mean PCE (%)', ascending=False)
        df_r_oh.to_csv(OUT_TABLE_OH_ALL, index=False)
        print(f"\nSaved OH Ranking to {OUT_TABLE_OH_ALL}")
        
    # FP Ranking
    if det_FP_k50 is not None:
        cids = sorted(df_B['linear_top3_k50'].unique())
        rr = []
        for cid in cids:
            st = get_stats(df_B, 'linear_top3_k50', cid, det_FP_k50, 'FP')
            rr.append([cid, st['Count'], st['Mean_PCE'], st['Std_Dev'], st['Max_PCE'], st['Major_Component']])
        df_r_fp = pd.DataFrame(rr, columns=['Cluster ID', 'N', 'Mean PCE (%)', 'Std Dev', 'Max PCE (%)', 'Major Component'])
        # Corrected sort column name (using correct space)
        df_r_fp = df_r_fp.sort_values('Mean PCE (%)', ascending=False)
        df_r_fp.to_csv(OUT_TABLE_FP_ALL, index=False)
        print(f"Saved FP Ranking to {OUT_TABLE_FP_ALL}")

else:
    print("[ERROR] Could not load main dataframes.")

Found Thesis A: None
Found Thesis B: None
Found Labels A: None
Found Labels B: None
[WARN] Missing files for merge: None, None
[WARN] Missing files for merge: None, None
[WARN] Detail file not found: Detail_OH_Clusters_linear_top3_k50.csv
[WARN] Detail file not found: Detail_OH_Clusters_linear_top3_k10.csv
[WARN] Detail file not found: Detail_FP_Clusters_linear_top3_k50.csv
[WARN] Detail file not found: Detail_FP_Clusters_linear_top3_k10.csv
[ERROR] Could not load main dataframes.
